<a href="https://colab.research.google.com/github/mozey256/OSCAAR/blob/main/Organized_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocess the dataset


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import preprocessing

In [ ]:
df = pd.read_csv('ML_training_data.csv')

In [ ]:
# Separate features (X) and target variable (y)
X = df.drop(columns=['Synergy'])  # Features
y = df['Synergy']  # Target variable

# Convert categorical labels to binary labels
y_binary = y.replace({'No': 0, 'Yes': 1})

In [ ]:
df.head()

In [ ]:
label_encoder = preprocessing.LabelEncoder()

df['Drug 1'] = label_encoder.fit_transform(df['Drug 1'])
df['Drug2'] = label_encoder.fit_transform(df['Drug2'])

In [ ]:
df['Synergy'] = df['Synergy'].replace({'No': 0, 'Yes': 1})

In [ ]:
df.drop(columns =['Combination'], inplace =True)

In [ ]:
df.head()

# Split and Balance the *data*

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

In [ ]:
X = df.drop(columns = ['Synergy'])
y= df['Synergy']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X , y , test_size=0.2, random_state =42)

balance the data

In [ ]:
smote = SMOTE(random_state =42)

In [ ]:
X_train1, y_train1 = smote.fit_resample(X_train, y_train)

In [ ]:
Xtrain , X_val, ytrain, y_val = train_test_split(X_train1, y_train1, test_size=0.2, random_state =42)

# Pycaret

In [ ]:
!pip install pycaret

In [ ]:
from pycaret.classification import *

In [ ]:
data = pd.concat([Xtrain, ytrain], axis =1)

In [ ]:
clf = setup(data, target ='Synergy')

In [ ]:
best = compare_models()

# Train and Test the model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
rf = RandomForestClassifier(random_state= 42)

rf. fit(Xtrain, ytrain)

print(classification_report(y_val, rf.predict(X_val)))
print(confusion_matrix(y_val, rf.predict(X_val)))

In [ ]:
print(classification_report(y_test, rf.predict(X_test)))
print(confusion_matrix(y_test, rf.predict(X_test)))

In [ ]:
important = rf.feature_importances_
selected = np.where(important >0)[0]

print(len(selected))
X_train_selected = Xtrain.iloc[:, selected]
X_test_selected =  X_test.iloc[:, selected]
X_val_selected = X_val.iloc[:, selected]

In [ ]:
rf_important = RandomForestClassifier(random_state= 10, n_estimators = 500 ,max_depth = 5, max_features = 'log2', oob_score= True)

rf_important. fit(X_train_selected, ytrain)

print(classification_report(y_val, rf_important.predict(X_val_selected)))
print(confusion_matrix(y_val, rf_important.predict(X_val_selected)))

In [ ]:
print(classification_report(y_test, rf_important.predict(X_test_selected)))
print(confusion_matrix(y_test, rf_important.predict(X_test_selected)))

In [ ]:
from sklearn.model_selection import GridSearchCV

# Create the random grid
random_grid = {
  'random_state': [10],
  'n_estimators': [1000, 500, 1000]
 }

rf1 = RandomForestClassifier(n_estimators = 500, random_state = 42, max_depth = 3, max_features = 'log2')
rf_random = GridSearchCV(rf1, random_grid, cv = 3)
# Fit the random search model
rf_random.fit(X_train_selected, ytrain)

In [ ]:
print(classification_report(y_val, rf_random.predict(X_val_selected)))
print(confusion_matrix(y_val, rf_random.predict(X_val_selected)))

In [ ]:

print(classification_report(y_test, rf_random.predict(X_test_selected)))
print(confusion_matrix(y_test, rf_random.predict(X_test_selected)))

# Gradient Booster

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbc = GradientBoostingClassifier(random_state= 42, n_estimators = 1000, learning_rate = 0.05)

gbc.fit(Xtrain, ytrain)

print(classification_report(y_val, gbc.predict(X_val)))
print(confusion_matrix(y_val, gbc.predict(X_val)))

In [ ]:
print(classification_report(y_test, gbc.predict(X_test)))
print(confusion_matrix(y_test, gbc.predict(X_test)))

In [ ]:
important_features = gbc.feature_importances_
importantF = np.where(important_features >0)[0]

Xtrain_selected =  Xtrain.iloc[:, importantF]
Xtest_selected = X_test.iloc[:, importantF]
Xval_selected = X_val.iloc[:, importantF]

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbc2 = GradientBoostingClassifier(random_state= 10, n_estimators = 500, learning_rate = 0.05)

gbc2.fit(Xtrain_selected, ytrain)

print(classification_report(y_val, gbc2.predict(Xval_selected)))
print(confusion_matrix(y_val, gbc2.predict(Xval_selected)))

In [ ]:
print(classification_report(y_test, gbc2.predict(Xtest_selected)))
print(confusion_matrix(y_test, gbc2.predict(Xtest_selected)))

In [ ]:
from sklearn.model_selection import GridSearchCV

# Create the random grid
random_grid = {
  'random_state': [10],
  'n_estimators': [1000, 500, 1000],
  'learning_rate': [0.00005,0.0005, 0.005],
  'max_depth': [2,3,4]
 }

gbc1 = GradientBoostingClassifier(n_estimators = 1000, random_state = 10, max_depth = 2, subsample = 0.5, learning_rate = 0.0005)
rf_random = GridSearchCV(gbc1, random_grid, cv = 4)
# Fit the random search model
rf_random.fit(Xtrain_selected, ytrain)

In [ ]:
print(classification_report(y_val, rf_random.predict(Xval_selected)))
print(confusion_matrix(y_val, rf_random.predict(Xval_selected)))

In [ ]:
print(classification_report(y_test, rf_random.predict(Xtest_selected)))
print(confusion_matrix(y_test, rf_random.predict(Xtest_selected)))

# *Ensemble  lr, dt, rf, gbc, svc , knn*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Initialize classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "SVM": SVC(),
    "k-NN": KNeighborsClassifier()
}

# Train and evaluate each classifier
for name, clf in classifiers.items():
    clf.fit(Xtrain, ytrain)
    y_pred = clf.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    clf = confusion_matrix(y_test, y_pred)

    # Print results
    print(f"Classifier: {name}")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print(clf)

In [ ]:
from sklearn.ensemble import VotingClassifier

# Define the individual classifiers
classifiers = [
    ("Logistic Regression", LogisticRegression()),
    ("Decision Tree", DecisionTreeClassifier()),
    ("Random Forest", RandomForestClassifier()),
    ("Gradient Boosting", GradientBoostingClassifier())
]

# Create the ensemble model
ensemble_model = VotingClassifier(classifiers)

# Train the ensemble model
ensemble_model.fit(Xtrain, ytrain)

# Evaluate the ensemble model
accuracy = ensemble_model.score(X_test, y_test)
print(f"Ensemble Model Accuracy: {accuracy:.2f}")


In [ ]:
print(classification_report(y_test, ensemble_model.predict(X_test)))
print(confusion_matrix(y_test, ensemble_model.predict(X_test)))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Convert categorical labels to binary labels
y_test_binary = y_test.replace({'No': 0, 'Yes': 1})

# Train the Random Forest classifier
rf_classifier = RandomForestClassifier()
rf_classifier.fit(X_train, y_train)

# Predict on test data
y_pred_rf = rf_classifier.predict(X_test)

# Calculate accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {accuracy_rf:.2f}")

# Calculate ROC curve and AUC score
if hasattr(rf_classifier, "predict_proba"):
    prob_pos_rf = rf_classifier.predict_proba(X_test)[:, 1]
else:  # use decision function
    prob_pos_rf = rf_classifier.decision_function(X_test)
    prob_pos_rf = (prob_pos_rf - prob_pos_rf.min()) / (prob_pos_rf.max() - prob_pos_rf.min())

fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test_binary, prob_pos_rf)
auc_rf = roc_auc_score(y_test_binary, prob_pos_rf)
print(f"Random Forest AUC: {auc_rf:.2f}")

# Plot ROC curve
plt.figure()
plt.plot(fpr_rf, tpr_rf, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % auc_rf)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (Random Forest)')
plt.legend(loc="lower right")
plt.show()



# New Section